# 03 — Prepare & Export

Add the share-of-whole columns the treemap story is built on, then package the analysis-ready
dataset (CSV + Excel + Parquet + codebook).

**Derived columns (all computed in DuckDB from `mcu_films_clean`):**
- `share_of_franchise` — a film's worldwide gross as a fraction of the **whole MCU** total.
  This is the treemap's tile size: it answers "how much of the franchise is this one film?"
- `share_of_saga` — worldwide gross as a fraction of the film's **Saga** total (Infinity vs
  Multiverse).
- `share_of_phase` — worldwide gross as a fraction of the film's **Phase** total.
- `intl_share` — international (non-U.S./Canada) gross as a fraction of worldwide, i.e.
  `1 - domestic_share`. How much of the film's take came from outside North America.
- `profit_multiple` — worldwide gross ÷ production budget (a rough "how many times its budget
  did it earn at the box office" — box-office only, ignores marketing and studio splits).

The dataset is one row per released theatrical MCU film.

In [ ]:
import sys, os
from pathlib import Path

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

import pandas as pd
from src.ingest import load_config
from src.clean_quality import get_connection, run_sql, load_to_duckdb, save_processed
from src.prepare import package_dataset

cfg = load_config('config.yaml')
con = get_connection(cfg)

## Compute share-of-whole columns

Window functions give each film its share of the franchise, its saga, and its phase in one
pass. Shares are rounded to 4 decimals (0.0001 = 0.01%).

In [ ]:
SQL = '''
SELECT
    title,
    phase_num,
    phase,
    saga,
    release_date,
    release_year,
    worldwide_gross,
    domestic_gross,
    international_gross,
    domestic_share,
    ROUND(1 - domestic_share, 4)                                              AS intl_share,
    ROUND(worldwide_gross::DOUBLE / SUM(worldwide_gross) OVER (), 4)           AS share_of_franchise,
    ROUND(worldwide_gross::DOUBLE / SUM(worldwide_gross) OVER (PARTITION BY saga), 4)  AS share_of_saga,
    ROUND(worldwide_gross::DOUBLE / SUM(worldwide_gross) OVER (PARTITION BY phase), 4) AS share_of_phase,
    opening_weekend,
    production_budget,
    ROUND(worldwide_gross::DOUBLE / production_budget, 2)                      AS profit_multiple
FROM mcu_films_clean
ORDER BY worldwide_gross DESC
'''
df = run_sql(SQL, con)
print(len(df), 'films')
df[['title','phase','saga','worldwide_gross','share_of_franchise','share_of_phase','intl_share','profit_multiple']].head(12)

Sanity checks: shares must sum to 1 at each level of aggregation (whole franchise = 1.0; each
saga's films sum to 1.0; each phase's films sum to 1.0).

In [ ]:
print('sum share_of_franchise:', round(df['share_of_franchise'].sum(), 3))
print('sum share_of_saga by saga:')
print(df.groupby('saga')['share_of_saga'].sum().round(3).to_string())
print('sum share_of_phase by phase:')
print(df.groupby('phase', sort=False)['share_of_phase'].sum().round(3).to_string())
assert abs(df['share_of_franchise'].sum() - 1.0) < 0.01
assert (df.groupby('saga')['share_of_saga'].sum().round(2) == 1.0).all()
assert (df.groupby('phase')['share_of_phase'].sum().round(2) == 1.0).all()
print('\nshare columns reconcile to 1.0 at every level')

### Phase- and saga-level roll-up (the treemap's group totals)

Quick look at the aggregates the treemap groups by — worldwide gross and franchise share per
Phase, which is where the "*Endgame*-era Phase 3 carries the franchise; Phase 4 fragments into
many smaller films" story shows up.

In [ ]:
rollup = run_sql('''
SELECT saga, phase,
       COUNT(*)                AS films,
       SUM(worldwide_gross)    AS worldwide_gross,
       ROUND(SUM(worldwide_gross)::DOUBLE / (SELECT SUM(worldwide_gross) FROM mcu_films_clean), 4) AS share_of_franchise,
       ROUND(AVG(worldwide_gross))  AS avg_film_gross
FROM mcu_films_clean
GROUP BY saga, phase, phase_num
ORDER BY phase_num
''', con)
rollup

## Save processed + package export

Write the film-level table to `data/processed/`, then package `mcu_box_office_v1` to
`export/` in all three formats with a plain-English codebook and a source/license note.

In [ ]:
load_to_duckdb(df, 'mcu_box_office', con)
save_processed(df, cfg, 'mcu_box_office.parquet')

In [ ]:
codebook = {
    'title':              'Film title (Marvel Cinematic Universe theatrical feature).',
    'phase_num':          'MCU Phase as an integer 1-6.',
    'phase':              'MCU Phase label ("Phase 1" ... "Phase 6"). Marvel Studios\' official release grouping.',
    'saga':               'Over-arching story arc: "Infinity Saga" (Phases 1-3) or "Multiverse Saga" (Phases 4-6).',
    'release_date':       'U.S. theatrical release date.',
    'release_year':       'Year of U.S. theatrical release.',
    'worldwide_gross':    'Lifetime worldwide theatrical box-office gross, in nominal (year-of-release) US dollars. Domestic + international.',
    'domestic_gross':     'Lifetime U.S. & Canada theatrical gross, nominal US dollars.',
    'international_gross': 'Lifetime gross outside the U.S. & Canada (worldwide minus domestic), nominal US dollars.',
    'domestic_share':     'Domestic gross as a fraction of worldwide (0-1).',
    'intl_share':         'International gross as a fraction of worldwide (0-1); equals 1 - domestic_share.',
    'share_of_franchise': 'This film\'s worldwide gross as a fraction of the ENTIRE MCU worldwide total (0-1). Sums to 1 across all films.',
    'share_of_saga':      'This film\'s worldwide gross as a fraction of its Saga total (0-1). Sums to 1 within each saga.',
    'share_of_phase':     'This film\'s worldwide gross as a fraction of its Phase total (0-1). Sums to 1 within each phase.',
    'opening_weekend':    'U.S. opening-weekend gross, nominal US dollars.',
    'production_budget':  'Reported/estimated production budget, nominal US dollars (excludes marketing).',
    'profit_multiple':    'Worldwide gross divided by production budget. Box-office-only ratio; ignores marketing spend and studio/exhibitor revenue splits, so it is NOT true profit.',
}
notes = '''
Source: The Numbers (the-numbers.com) Marvel Cinematic Universe franchise page, per-film
box office; cross-checked against Box Office Mojo (domestic figures agree to within ~1.4%).
Retrieved 2026-09-19.

Scope: released THEATRICAL MCU feature films only (Marvel Studios Phase 1-6 canon, including
the Sony-distributed Tom Holland Spider-Man films). Excludes unreleased/future films, TV
specials, and Disney+ series.

All dollar figures are NOMINAL (year-of-release) and NOT inflation-adjusted; cross-era raw
gross comparisons therefore understate older films. This dataset\'s purpose is share-of-
franchise composition, not cross-era ranking. License: box-office figures are cited from an
industry aggregator for a non-commercial pop-culture project; see SOURCES.md.
'''
written = package_dataset(df, cfg, name='mcu_box_office_v1', codebook=codebook, notes=notes)
list(written.items())

---
**Next:** `04-viz.ipynb` — explore the treemap and share-of-whole cuts (matplotlib) to
settle the story before building publication charts in `06-viz-social`.

---
## Cleanup
Close the DuckDB connection so the single-writer lock is released.

In [ ]:
con.close()
print('connection closed')